# DGCA domestic airlines: interactive Plotly dashboard
Fourth notebook. Reads the cleaned table from `data/clean/` and writes **one self-contained HTML file** to `outputs/dashboard/`. Plotly.js is embedded in the file, so it opens in any browser without internet, which is useful for the in-person presentation.

What the page contains:
- four headline cards, an **airline picker** that updates a summary and highlights that airline in the line charts,
- seven interactive figures: market total, airline by airline (two views), a month-by-month play/pause race, market share, load factor (two views), a gain/loss scorecard (two views), and a before/after-cleaning comparison,
- a caption with numbers under every figure, a short "How we counted" box, and sources.

Numbers are stated in everyday units: passengers per month in millions, "out of every 100 passengers", "out of every 100 seats", and the same month of 2019 as the yardstick.

In [1]:
%pip install -q plotly pyarrow pandas numpy


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Setup and data
Group-level monthly series for scheduled domestic flights, cut at the last month where the airlines add up to DGCA's own total.

In [2]:
# 1. SETUP AND DATA: group-level monthly series for scheduled domestic traffic
import json
from pathlib import Path
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from plotly.offline import get_plotlyjs

TODAY = pd.Timestamp.today().date().isoformat()
_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents] if (p / "data" / "clean").is_dir())
RAW, INTERIM, CLEAN = (PROJECT_ROOT / "data" / d for d in ("raw", "interim", "clean"))
OUT = PROJECT_ROOT / "outputs" / "dashboard"; OUT.mkdir(parents=True, exist_ok=True)
latest = lambda folder, pat: sorted(folder.glob(pat))[-1]

clean = pd.read_parquet(latest(CLEAN, "dgca_domestic_clean_*.parquet"))
interim = pd.read_csv(latest(INTERIM, "dgca_domestic_airline_monthly_*.csv"))
rec = pd.read_csv(latest(CLEAN, "total_domestic_reconciliation_*.csv"), parse_dates=["Date"])
LAST = rec.loc[rec.within_2pct, "Date"].max()                 # last month where the airlines add up to DGCA's own total
START = pd.Timestamp("2019-01-01")
HL, DARK, MID, LIGHT = "#d95f02", "#404040", "#8c8c8c", "#d0d0d0"     # one highlight colour, the rest grey
MIN_BASE_SHARE = 0.005          # a group needs >= 0.5% of 2019 passengers for a fair 2019 comparison
FONT = "system-ui, -apple-system, 'Segoe UI', Roboto, Arial, sans-serif"

fm = lambda d: pd.Timestamp(d).strftime("%b %Y")
iso = lambda idx: [pd.Timestamp(d).strftime("%Y-%m-%d") for d in idx]
lst = lambda s, k=4: [None if pd.isna(v) else round(float(v), k) for v in s]

sdc = clean[(clean.Type == "ScheduledDomestic") & (clean.Date <= LAST)]
months = pd.date_range("2017-01-01", LAST, freq="MS")
piv = lambda col: sdc.groupby(["Group", "Date"])[col].sum(min_count=1).unstack("Group").reindex(months)
pax, pkp, skp = piv("Passenger Number"), piv("Passenger Kilometers"), piv("Seat Kilometers")

paxf = pax.copy()               # assumption: a group with no row after it started has stopped = 0 passengers
for g in paxf.columns:
    f = paxf[g].first_valid_index()
    if f is not None:
        paxf.loc[(paxf.index >= f) & paxf[g].isna(), g] = 0
total = paxf.sum(axis=1)

b19 = pax[pax.index.year == 2019]
pos12, share19 = (b19 > 0).sum(), b19.sum() / b19.sum().sum()
keep = [g for g in pax.columns if pos12[g] == 12 and share19[g] >= MIN_BASE_SHARE]
excluded = {g: ("no 2019 traffic" if b19[g].sum() == 0 else f"only {pos12[g]} months in 2019" if pos12[g] < 12 else "too small in 2019")
            for g in pax.columns if g not in keep and paxf[g].loc[START:].max() >= 40_000}
base = b19[keep].copy(); base.index = base.index.month
ratio = paxf[keep].div(base.reindex(paxf.index.month).set_axis(paxf.index))

# groups worth drawing: at least 40,000 passengers in some month since 2019, biggest in 2019 first
show = [g for g in paxf.columns if paxf[g].loc[START:].max() >= 40_000]
show.sort(key=lambda g: (-b19[g].sum(), -paxf[g].iloc[-3:].mean()))
big = [g for g in show if paxf[g].loc[START:].max() >= 500_000 and g != "Jet Airways"]   # drawn by default in the passenger view
print("airline groups drawn:", show); print("excluded from the 2019 comparison:", excluded)

airline groups drawn: ['IndiGo', 'Air India Group', 'SpiceJet', 'Go First', 'Jet Airways', 'Alliance Air', 'TruJet', 'Star Air', 'Akasa Air', 'Fly91']
excluded from the 2019 comparison: {'Akasa Air': 'no 2019 traffic', 'Fly91': 'no 2019 traffic', 'Jet Airways': 'only 4 months in 2019', 'Star Air': 'too small in 2019'}


## 2. Numbers for the cards, the airline picker and the captions

In [3]:
# 2. NUMBERS FOR THE CARDS, THE AIRLINE PICKER AND THE CAPTIONS
HL_GROUP = "IndiGo"
n3 = slice(-3, None)
last3 = paxf.index[n3]

# market-level
tot19_by_month = paxf[paxf.index.year == 2019].sum(axis=1); tot19_by_month.index = tot19_by_month.index.month
ratio_total = total / tot19_by_month.reindex(total.index.month).values
roll_total = ratio_total.rolling(3).mean()
hit = roll_total.loc["2020-07-01":][roll_total.loc["2020-07-01":] >= 1.0]
market_back = hit.index[0] if len(hit) else None
avg19 = tot19_by_month.mean() / 1e6
trough_val, trough_date = total.loc[START:].min() / 1e6, total.loc[START:].idxmin()
l12 = total.iloc[-12:].mean() / 1e6
latest_m = total.index[-1]
same_m_2019 = total[(total.index.year == 2019) & (total.index.month == latest_m.month)].iloc[0] / 1e6

# per group
res_rows = {}
for g in keep:
    roll = ratio[g].rolling(3).mean().loc["2020-07-01":]; h = roll[roll >= 1.0]
    res_rows[g] = dict(recovered=h.index[0] if len(h) else None, latest_pct=ratio[g].iloc[n3].mean() * 100)

def lf_between(g, sl):
    a, b = pkp[g].loc[sl].sum(), skp[g].loc[sl].sum()
    return a / b * 100 if b > 0 else np.nan
sh_all = paxf.div(total, axis=0) * 100
stats = {}
for g in show:
    last_tr = paxf[g][paxf[g] > 0].index.max(); first_tr = paxf[g][paxf[g] > 0].index.min()
    stopped = last_tr <= LAST - pd.Timedelta(days=62)
    now = paxf[g].iloc[n3].mean() / 1e6
    s = dict(now_m=round(now, 2), share19=round(share19.get(g, 0) * 100, 1), share_now=round(sh_all[g].iloc[n3].mean(), 1),
             lf19=None, lf_now=None, avg19_m=None, pct=None, back=None)
    if g in keep:
        s["avg19_m"] = round(b19[g].mean() / 1e6, 2); s["pct"] = round(res_rows[g]["latest_pct"])
        s["back"] = fm(res_rows[g]["recovered"]) if res_rows[g]["recovered"] is not None else "never"
    l19, lnow = lf_between(g, slice("2019-01-01", "2019-12-01")), lf_between(g, slice(paxf.index[-12], None))
    s["lf19"] = None if np.isnan(l19) else round(l19); s["lf_now"] = None if np.isnan(lnow) else round(lnow)
    if g in keep and stopped:
        s["sentence"] = f"Stopped flying after {fm(last_tr)}. It never got back to its 2019 passenger numbers."
    elif g in keep and res_rows[g]["recovered"] is not None and s["pct"] >= 100:
        s["sentence"] = f"Back above its 2019 passenger numbers since {s['back']}. Now carries {s['pct']}% of what it carried in the same months of 2019."
    elif g in keep and res_rows[g]["recovered"] is not None:
        s["sentence"] = f"Reached its 2019 level in {s['back']}, but has since slipped to {s['pct']}% of 2019."
    elif g in keep:
        s["sentence"] = f"Still flying, but carries only {s['pct']}% of what it carried in 2019. It has never got back to 2019."
    elif stopped:
        s["sentence"] = f"Stopped flying after {fm(last_tr)} (it had no full 2019 year to compare with)."
    elif b19[g].sum() == 0:
        s["sentence"] = f"Started flying after 2019 (first passengers: {fm(first_tr)}), so there is no 2019 level to compare with. Now carries {now:.2f} million passengers a month."
    else:
        s["sentence"] = "Too small in 2019 for a fair comparison with 2019."
    stats[g] = s

# cards
big_back = [g for g in keep if res_rows[g]["latest_pct"] >= 100]
top = max(show, key=lambda g: sh_all[g].iloc[n3].mean())
CARDS = [
    (f"{total.iloc[-1] / 1e6:.1f} million", f"passengers flew in {latest_m:%B %Y}", f"{same_m_2019:.1f} million in {latest_m:%B} 2019 ({(total.iloc[-1] / 1e6 / same_m_2019 - 1) * 100:+.0f}%)"),
    (fm(market_back) if market_back is not None else "not yet", "market back to its 2019 level", "first month the 3-month average matched the same months of 2019"),
    (f"{sh_all[top].iloc[n3].mean():.0f} in 100", f"passengers fly {top} today", f"it was {share19.get(top, 0) * 100:.0f} in 100 in 2019"),
    (f"{len(big_back)} of {len(keep)}", "airline groups that flew all of 2019 are back above it", ", ".join(big_back) if big_back else "none"),
]
print(CARDS); print({g: stats[g]["sentence"] for g in ["IndiGo", "SpiceJet", "Go First", "Akasa Air"] if g in stats})

[('12.1 million', 'passengers flew in August 2026', '11.8 million in August 2019 (+3%)'), ('Feb 2023', 'market back to its 2019 level', 'first month the 3-month average matched the same months of 2019'), ('66 in 100', 'passengers fly IndiGo today', 'it was 47 in 100 in 2019'), ('2 of 6', 'airline groups that flew all of 2019 are back above it', 'Air India Group, IndiGo')]
{'IndiGo': 'Back above its 2019 passenger numbers since May 2022. Now carries 146% of what it carried in the same months of 2019.', 'SpiceJet': 'Still flying, but carries only 11% of what it carried in 2019. It has never got back to 2019.', 'Go First': 'Stopped flying after May 2023. It never got back to its 2019 passenger numbers.', 'Akasa Air': 'Started flying after 2019 (first passengers: Aug 2022), so there is no 2019 level to compare with. Now carries 0.73 million passengers a month.'}


## 3. The seven figures

In [4]:
# 3. THE SEVEN FIGURES (Plotly). Every figure is a plain Plotly object; the page itself is assembled in the next cell.
CFG = dict(displaylogo=False, responsive=True, modeBarButtonsToRemove=["lasso2d", "select2d", "autoScale2d"])
def style(fig, h=470, legend=True):
    fig.update_layout(template="plotly_white", font=dict(family=FONT, size=13, color="#222"), height=h,
                      margin=dict(l=64, r=40, t=36, b=60), hoverlabel=dict(font_size=13),
                      legend=dict(orientation="h", y=-0.22, x=0) if legend else None, showlegend=legend)
    return fig
WAVES = [("2020-03-01", "2020-06-01", "lockdown"), ("2021-04-01", "2021-07-01", "2nd wave")]
def waves(fig, **kw):
    for a, b, lab in WAVES:
        fig.add_vrect(x0=a, x1=b, fillcolor=LIGHT, opacity=.45, layer="below", line_width=0, **kw)
        fig.add_annotation(x=a, y=1.0, xref="x", yref="paper", text=lab, showarrow=False, xanchor="left", yanchor="bottom", font=dict(size=11, color=MID))
DATES = iso(paxf.loc[START:].index)
mon = lambda d: pd.Timestamp(d).strftime("%b %Y")

# ---- 1. the whole market
t = total.loc[START:] / 1e6
f1 = go.Figure(go.Scatter(x=DATES, y=lst(t, 3), mode="lines", line=dict(color=DARK, width=3), name="Passengers",
        hovertext=[f"<b>{mon(d)}</b><br>{v:.1f} million passengers<br>{v / avg19 * 100:.0f}% of the 2019 monthly average" for d, v in zip(t.index, t)], hoverinfo="text"))
f1.add_hline(y=avg19, line_dash="dash", line_color="black", line_width=1, annotation_text=f"2019 average: {avg19:.1f} million a month", annotation_position="top left", annotation_font_size=12)
f1.add_trace(go.Scatter(x=[iso([trough_date])[0]], y=[trough_val], mode="markers+text", marker=dict(color=HL, size=13), text=[f"{mon(trough_date)}: {trough_val:.1f} million"],
        textposition="top right", hoverinfo="skip", name="Lowest month", showlegend=False))
waves(f1)
rng = lambda a, b: dict(label=f"{a}", method="relayout", args=[{"xaxis.range": b}])
f1.update_layout(xaxis=dict(rangeslider=dict(visible=True, thickness=.07), range=[DATES[0], DATES[-1]]),
                 yaxis=dict(title="Passengers per month (millions)", rangemode="tozero"),
                 updatemenus=[dict(type="buttons", direction="right", x=0, y=1.16, xanchor="left", buttons=[
                     rng("All years", [DATES[0], DATES[-1]]), rng("2019 to 2021", ["2019-01-01", "2021-12-31"]), rng("2022 onwards", ["2022-01-01", DATES[-1]])])])
style(f1, 440, legend=False)

# ---- 2. airline by airline (two views, switched by the buttons)
LINEMETA = lambda g: dict(kind="line", group=g)
def line(g, x, y, hover, visible, showlegend, name=None):
    on = g == HL_GROUP
    txt = [""] * (len(y) - 1) + [g]
    return go.Scatter(x=x, y=y, mode="lines+text", name=name or g, meta=LINEMETA(g), visible=visible, showlegend=showlegend, hovertext=hover, hoverinfo="text",
                      line=dict(color=HL if on else MID, width=4 if on else 1.6), opacity=1 if on else .6,
                      text=txt, textposition="middle right", textfont=dict(color=HL if on else MID, size=11), connectgaps=False)
b19m = pax[pax.index.year == 2019].copy(); b19m.index = b19m.index.month
traces = []
for g in show:
    v = paxf[g].loc[START:] / 1e6
    hv = []
    for d, x in zip(v.index, v):
        s = f"<b>{g}</b><br>{mon(d)}: {x:.2f} million passengers"
        if g in keep:
            r = ratio[g].loc[d]; s += f"<br>{r * 100:.0f}% of {pd.Timestamp(d):%B} 2019"
        hv.append(s)
    traces.append(line(g, DATES, lst(v, 3), hv, True if g in big else "legendonly", True))
n_p = len(show)
for g in keep:
    r = ratio[g].loc[START:] * 100
    hv = [f"<b>{g}</b><br>{mon(d)}: {x:.0f}% of {pd.Timestamp(d):%B} 2019<br>({paxf[g].loc[d] / 1e6:.2f} million passengers vs {b19m[g].loc[d.month] / 1e6:.2f} million in 2019)" for d, x in zip(r.index, r)]
    traces.append(line(g, DATES, lst(r, 1), hv, False, False))
n_r = len(keep)
traces.append(go.Scatter(x=DATES, y=[100] * len(DATES), mode="lines", line=dict(color="black", dash="dash", width=1), name="Same as 2019 (100%)", visible=False, showlegend=False,
                         hovertemplate="Same as the same month of 2019<extra></extra>", meta=dict(kind="ref")))
f2 = go.Figure(traces)
vis_pax = [True if g in big else "legendonly" for g in show] + [False] * n_r + [False]
vis_pct = [False] * n_p + [True] * n_r + [True]
f2.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0, y=1.16, xanchor="left", buttons=[
    dict(label="Passengers per month", method="update", args=[{"visible": vis_pax, "showlegend": [True] * n_p + [False] * (n_r + 1)}, {"yaxis.title.text": "Passengers per month (millions)", "yaxis.autorange": True}]),
    dict(label="Compared with the same month of 2019", method="update", args=[{"visible": vis_pct, "showlegend": [False] * n_p + [True] * n_r + [False]}, {"yaxis.title.text": "% of the same month of 2019 (100 = same as 2019)", "yaxis.range": [0, 200], "yaxis.autorange": False}])])],
    xaxis=dict(range=[DATES[0], iso([LAST + pd.DateOffset(months=9)])[0]]), yaxis=dict(title="Passengers per month (millions)", rangemode="tozero"))
waves(f2)
for g in ["Go First"]:
    lt = paxf[g][paxf[g] > 0].index.max()
    f2.add_annotation(x=iso([lt])[0], y=0, text=f"{g} stops flying<br>({mon(lt)})", showarrow=True, arrowhead=2, ax=0, ay=-55, font=dict(size=11, color=DARK))
style(f2, 500)

# ---- 3. month-by-month race: bars = this month, black tick = the same month of 2019
race = [g for g in show if paxf[g].loc["2020-01-01":].max() >= 40_000]
race.sort(key=lambda g: (-paxf[g].iloc[n3].mean(), -b19[g].sum()))
rm = paxf.loc["2020-01-01":].index
def frame_data(d):
    vals = [paxf[g].loc[d] / 1e6 for g in race]
    ticks = [(b19m[g].loc[d.month] / 1e6 if pd.notna(b19m[g].loc[d.month]) and b19m[g].loc[d.month] > 0 else None) for g in race]
    cols = [LIGHT if tk is None else (HL if v >= tk else MID) for v, tk in zip(vals, ticks)]
    hov = [f"<b>{g}</b><br>{mon(d)}: {v:.2f} million passengers" + (f"<br>same month of 2019: {tk:.2f} million" if tk else "<br>no 2019 level to compare") for g, v, tk in zip(race, vals, ticks)]
    bar = go.Bar(x=vals, y=race, orientation="h", marker=dict(color=cols, line=dict(color=DARK, width=.6)), text=[f"{v:.1f} M" for v in vals], textposition="outside", hovertext=hov, hoverinfo="text", showlegend=False)
    tick = go.Scatter(x=ticks, y=race, mode="markers", marker=dict(symbol="line-ns", size=26, line=dict(color="black", width=3)), hoverinfo="skip", showlegend=False)
    return bar, tick
frames = []
for d in rm:
    bar, tick = frame_data(d)
    frames.append(go.Frame(data=[bar, tick], traces=[0, 1], name=mon(d), layout=go.Layout(title=dict(text=f"<b>{pd.Timestamp(d):%B %Y}</b>", x=0.01, font=dict(size=22, color=DARK)))))
b0, k0 = frame_data(rm[0])
legend_dummies = [go.Bar(x=[None], y=[None], orientation="h", name="Above the same month of 2019", marker=dict(color=HL, line=dict(color=DARK, width=.6))),
                  go.Bar(x=[None], y=[None], orientation="h", name="Below the same month of 2019", marker=dict(color=MID, line=dict(color=DARK, width=.6))),
                  go.Bar(x=[None], y=[None], orientation="h", name="Started after 2019 (nothing to compare)", marker=dict(color=LIGHT, line=dict(color=DARK, width=.6))),
                  go.Scatter(x=[None], y=[None], mode="markers", name="Same month of 2019", marker=dict(symbol="line-ns", size=14, line=dict(color="black", width=3)))]
f3 = go.Figure(data=[b0, k0] + legend_dummies, frames=frames)
vmax = max(paxf[g].loc["2020-01-01":].max() for g in race) / 1e6
step = lambda n: dict(method="animate", label=n, args=[[n], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))])
f3.update_layout(xaxis=dict(range=[0, vmax * 1.15], title="Passengers in the month (millions)"), yaxis=dict(autorange="reversed", automargin=True),
                 title=dict(text=f"<b>{mon(rm[0])}</b>", x=0.01, font=dict(size=22, color=DARK)),
                 updatemenus=[dict(type="buttons", direction="left", x=0, y=-0.34, xanchor="left", yanchor="top", pad=dict(t=0), buttons=[
                     dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=380, redraw=True), fromcurrent=True, transition=dict(duration=120))]),
                     dict(label="❚❚ Pause", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])],
                 sliders=[dict(active=0, x=0.14, len=0.86, y=-0.28, currentvalue=dict(visible=False), pad=dict(t=0), steps=[step(mon(d)) for d in rm], ticklen=0, font=dict(size=10))],
                 barmode="overlay")
style(f3, 560); f3.update_layout(legend=dict(orientation="h", y=-0.12, x=0), margin=dict(l=130, r=50, t=60, b=150))
f3.update_layout(title=dict(text=f"<b>{mon(rm[0])}</b>", x=0.01, font=dict(size=22, color=DARK)))

# ---- 4. who carries the passengers
order = [g for g in ["IndiGo", "Air India Group", "SpiceJet", "Go First", "Akasa Air"] if g in paxf.columns]
shs = sh_all.loc[START:]
shs = shs.where(total.loc[START:] > 0)               # no share in months with no flights at all
other = 100 - shs[order].sum(axis=1)
cols4 = [HL, DARK, MID, "#b5b5b5", "#d6d6d6"][:len(order)] + ["#ececec"]
f4 = go.Figure()
for g, c in zip(order + ["All other airlines"], cols4):
    y = shs[g] if g in order else other
    f4.add_trace(go.Scatter(x=DATES, y=lst(y, 2), mode="lines", stackgroup="one", name=g, line=dict(width=.6, color="white"), fillcolor=c,
                            hovertemplate=f"{g}: %{{y:.0f}} of every 100 passengers<extra></extra>"))
waves(f4)
f4.update_layout(hovermode="x unified", yaxis=dict(title="Passengers (out of every 100)", range=[0, 100]), xaxis=dict(range=[DATES[0], DATES[-1]]))
style(f4, 470)

# ---- 5. are the planes as full as before? (two views: monthly, or smoothed over 12 months)
lfg = [g for g in keep + ["Akasa Air"] if g in pkp.columns]
mon_lf = (pkp / skp.where(skp > 0) * 100).loc[START:]
yr_lf = (pkp.rolling(12, min_periods=12).sum() / skp.rolling(12, min_periods=12).sum().where(lambda s: s > 0) * 100).loc[START:]
traces5 = []
for mode, tbl, vis in [("12", yr_lf, True), ("1", mon_lf, False)]:
    for g in lfg:
        s = tbl[g]
        hv = [f"<b>{g}</b><br>{mon(d)}: {v:.0f} of every 100 seats filled" + ("" if mode == "1" else " (12-month average)") for d, v in s.items()]
        traces5.append(line(g, DATES, lst(s, 1), hv, vis, vis))
f5 = go.Figure(traces5); nl = len(lfg)
f5.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0, y=1.16, xanchor="left", buttons=[
    dict(label="12-month average (smoother)", method="update", args=[{"visible": [True] * nl + [False] * nl, "showlegend": [True] * nl + [False] * nl}]),
    dict(label="Each month", method="update", args=[{"visible": [False] * nl + [True] * nl, "showlegend": [False] * nl + [True] * nl}])])],
    xaxis=dict(range=[DATES[0], iso([LAST + pd.DateOffset(months=9)])[0]]), yaxis=dict(title="Seats filled (out of every 100)", range=[30, 100]))
waves(f5); style(f5, 470)

# ---- 6. scorecard: change in passengers per month (millions) or % of 2019
rows6 = []
for g in keep:
    now = paxf[g].iloc[n3].mean() / 1e6; b = np.mean([b19m[g].loc[d.month] for d in last3]) / 1e6
    rows6.append((g, now, b, now - b, now / b * 100))
rows6.sort(key=lambda r: -r[3]); names6 = [r[0] for r in rows6]
hov6 = [f"<b>{g}</b><br>{now:.2f} million passengers a month now<br>{b:.2f} million in the same months of 2019" for g, now, b, d, p in rows6]
bar_abs = go.Bar(x=[r[3] for r in rows6], y=names6, orientation="h", marker=dict(color=[HL if r[3] >= 0 else MID for r in rows6], line=dict(color=DARK, width=.6)),
                 text=[f"{r[3]:+.1f} M" for r in rows6], textposition="outside", hovertext=hov6, hoverinfo="text", showlegend=False)
bar_pct = go.Bar(x=[r[4] for r in rows6], y=names6, orientation="h", marker=dict(color=[HL if r[4] >= 100 else MID for r in rows6], line=dict(color=DARK, width=.6)),
                 text=[f"{r[4]:.0f}%" for r in rows6], textposition="outside", hovertext=hov6, hoverinfo="text", showlegend=False, visible=False)
f6 = go.Figure([bar_abs, bar_pct])
f6.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0, y=1.2, xanchor="left", buttons=[
    dict(label="Passengers gained or lost per month", method="update", args=[{"visible": [True, False]}, {"xaxis.title.text": "Passengers per month now minus the same months of 2019 (millions)", "xaxis.autorange": True, "shapes[0].visible": False}]),
    dict(label="% of what it carried in 2019", method="update", args=[{"visible": [False, True]}, {"xaxis.title.text": "% of the same months of 2019 (100 = same as 2019)", "xaxis.range": [0, 175], "shapes[0].visible": True}])])],
    shapes=[dict(type="line", x0=100, x1=100, xref="x", y0=0, y1=1, yref="paper", line=dict(color="black", dash="dash", width=1), visible=False)],
    xaxis=dict(title="Passengers per month now minus the same months of 2019 (millions)", zeroline=True, zerolinewidth=2, zerolinecolor="black", automargin=True),
    yaxis=dict(autorange="reversed", automargin=True), margin=dict(l=130))
style(f6, 380, legend=False); f6.update_layout(margin=dict(l=130, r=60, t=70, b=60))

# ---- 7. before vs after cleaning
core = ["Aircraft Number", "Aircraft Hours", "Aircraft Kilometres", "Passenger Number", "Passenger Kilometers", "Seat Kilometers", "Passenger Load Factor"]
na_b, na_a = interim[core].isna().mean() * 100, clean[core].isna().mean() * 100
brands = ["Air India", "Air India Express", "Vistara", "AirAsia India", "AIX Connect"]
sdf = clean[clean.Type == "ScheduledDomestic"]
bp = sdf[sdf.Airline.isin(brands)].groupby(["Airline", "Date"])["Passenger Number"].sum().unstack("Airline") / 1e6
gp = sdf[sdf.Group == "Air India Group"].groupby("Date")["Passenger Number"].sum() / 1e6
x7 = pd.Timestamp("2022-01-01")
f7 = make_subplots(rows=1, cols=2, column_widths=[.42, .58], subplot_titles=("Empty cells per column (% of rows)", "Air India Group: brands separate vs combined"), horizontal_spacing=.14)
f7.add_trace(go.Bar(y=core, x=na_b.values, orientation="h", name=f"Before cleaning ({len(interim):,} rows)", marker=dict(color=MID), hovertemplate="%{y}: %{x:.1f}% empty<extra>before</extra>"), 1, 1)
f7.add_trace(go.Bar(y=core, x=na_a.values, orientation="h", name=f"After cleaning ({len(clean):,} rows)", marker=dict(color=HL), hovertemplate="%{y}: %{x:.1f}% empty<extra>after</extra>"), 1, 1)
for c in brands:
    s = bp[c].loc[x7:LAST].dropna()
    f7.add_trace(go.Scatter(x=iso(s.index), y=lst(s, 3), mode="lines", name=c, line=dict(color=MID, width=1.4), hovertemplate=f"{c}: %{{y:.2f}} million<extra></extra>", legendgroup="brands"), 1, 2)
s = gp.loc[x7:LAST]
f7.add_trace(go.Scatter(x=iso(s.index), y=lst(s, 3), mode="lines", name="Air India Group (combined)", line=dict(color=HL, width=4), hovertemplate="Air India Group: %{y:.2f} million<extra></extra>"), 1, 2)
f7.update_yaxes(autorange="reversed", row=1, col=1); f7.update_yaxes(title_text="Passengers per month (millions)", row=1, col=2)
f7.update_xaxes(title_text="Empty cells (% of rows)", row=1, col=1)
f7.update_layout(barmode="group"); style(f7, 440); f7.update_layout(margin=dict(l=150, r=40, t=60, b=80), legend=dict(orientation="h", y=-0.18))
FIGS = dict(market=f1, recovery=f2, race=f3, share=f4, lf=f5, score=f6, quality=f7)
print({k: len(v.data) for k, v in FIGS.items()}, "| race frames:", len(frames))

{'market': 2, 'recovery': 17, 'race': 6, 'share': 6, 'lf': 14, 'score': 2, 'quality': 8} | race frames: 80


## 4. Captions and the HTML page

In [5]:
# 4. CAPTIONS (numbers computed from the data) AND THE HTML PAGE
S = stats
meta = json.load(open(latest(INTERIM, "collection_metadata_*.json")))
last_rm = rm[-1]
race_above = [g for g in race if g in keep and paxf[g].loc[last_rm] >= b19m[g].loc[last_rm.month]]
race_cmp = [g for g in race if g in keep]
gain = sum(r[3] for r in rows6 if r[3] > 0); loss = -sum(r[3] for r in rows6 if r[3] < 0)
ask_chg = (skp["SpiceJet"].iloc[-12:].sum() / skp["SpiceJet"].loc["2019-01-01":"2019-12-01"].sum() - 1) * 100
zero_rows = int((clean["Passenger Number"].eq(0) & clean["Passenger Kilometers"].eq(0) & clean["Seat Kilometers"].eq(0) & clean["Passenger Load Factor"].isna()).sum())
o24, d24 = pd.Timestamp("2024-10-01"), pd.Timestamp("2024-12-01")
sg = lambda x: f"{x:+.1f}".replace("-", "\u2212")
CAP = {
 "market": f"In 2019 about {avg19:.1f} million people a month took a scheduled domestic flight. In {mon(trough_date)} that fell to {"zero" if trough_val < 0.05 else f"{trough_val:.1f} million"}, and the market first matched its 2019 level again in {fm(market_back)}. "
           f"Over the last 12 months it averaged {l12:.1f} million a month, {l12 / avg19 * 100:.0f}% of 2019.",
 "recovery": f"{HL_GROUP} now carries {S[HL_GROUP]['now_m']:.1f} million passengers a month against {S[HL_GROUP]['avg19_m']:.1f} million in 2019 ({S[HL_GROUP]['pct']}%), and Air India Group {S['Air India Group']['now_m']:.1f} million against {S['Air India Group']['avg19_m']:.1f} million ({S['Air India Group']['pct']}%). "
             f"SpiceJet is at {S['SpiceJet']['pct']}% of 2019 and Go First stopped flying after {fm(paxf['Go First'][paxf['Go First'] > 0].index.max())}.",
 "race": f"In {last_rm:%B %Y}, {len(race_above)} of the {len(race_cmp)} groups that flew in the same month of 2019 are above their black tick ({', '.join(race_above)}). "
         f"Go First's bar vanishes after May 2023. Akasa Air, which began in August 2022, has no 2019 tick, and Star Air's tick is its launch year, so neither is counted.",
 "share": f"{top} carried {share19.get(top, 0) * 100:.0f} of every 100 passengers in 2019 and {sh_all[top].iloc[n3].mean():.0f} of every 100 in the latest three months. "
          f"Air India Group moved from {S['Air India Group']['share19']:.0f} to {S['Air India Group']['share_now']:.0f}, SpiceJet from {S['SpiceJet']['share19']:.0f} to {S['SpiceJet']['share_now']:.0f}, and Akasa Air went from 0 to {S['Akasa Air']['share_now']:.0f}.",
 "lf": f"Planes are about as full as before, a little less for IndiGo. Over the last 12 months {HL_GROUP} filled {S[HL_GROUP]['lf_now']} of every 100 seats ({S[HL_GROUP]['lf19']} in 2019) and Air India Group {S['Air India Group']['lf_now']} ({S['Air India Group']['lf19']}). "
       f"SpiceJet filled {S['SpiceJet']['lf_now']} ({S['SpiceJet']['lf19']} in 2019): somewhat emptier planes, but the bigger story is that it now flies {abs(ask_chg):.0f}% fewer seat-kilometres than in 2019.",
 "score": f"The groups that grew added {gain:.1f} million passengers a month compared with the same months of 2019, led by {rows6[0][0]} ({sg(rows6[0][3])} million). "
          f"The groups that shrank or stopped lost {loss:.1f} million a month, the most by {rows6[-1][0]} ({sg(rows6[-1][3])} million). Akasa Air (about {S['Akasa Air']['now_m']:.1f} million a month, started 2022) is not shown because it has no 2019 level.",
 "quality": f"Left: removing the empty template rows cuts the empty share of Passenger Number from {na_b['Passenger Number']:.1f}% to {na_a['Passenger Number']:.1f}%; {zero_rows} of the {int(clean['Passenger Load Factor'].isna().sum())} empty load factors are months with zero flying, where 0/0 is undefined. "
            f"Right: after Vistara's last month the Air India brand alone jumps from {bp['Air India'].get(o24, np.nan):.2f} to {bp['Air India'].get(d24, np.nan):.2f} million, while the combined group stays continuous ({gp.get(o24, np.nan):.2f} to {gp.get(d24, np.nan):.2f} million).",
}
HOW = {
 "market": "Each point is one month. The dashed line is the 2019 monthly average, and the grey bands mark the lockdown and the second wave. Use the buttons or the slider under the chart to zoom into a period.",
 "recovery": "Each line is one airline group. Choose an airline in the bar at the top to make it orange. Use the buttons to switch between real passenger numbers and a comparison with the same month of 2019, and click names in the legend to add or remove groups.",
 "race": "Press play, or drag the slider. Each bar is how many passengers the group carried that month; the black tick is what it carried in the same month of 2019. Orange means it has passed that tick.",
 "share": "Read it as 100 passengers. The orange layer at the bottom is the share that flies IndiGo. Hover to see the split for any month.",
 "lf": "A load factor is the share of seats that were filled. Use the buttons to switch between each month and a 12-month average. Choose an airline at the top to highlight it.",
 "score": "One bar per group that flew all of 2019 and was big enough to compare. The comparison uses the latest three months against the same three months of 2019.",
 "quality": "The same data before and after cleaning. Hover over the bars and lines for the exact values.",
}
TITLES = {"market": "How many people fly?", "recovery": "Airline by airline: who is back?", "race": "Watch it month by month",
          "share": "Who carries the passengers?", "lf": "Are the planes as full as before?", "score": "Scorecard: who gained and who lost?", "quality": "Data quality: before and after cleaning"}
divs = {k: pio.to_html(v, full_html=False, include_plotlyjs=False, div_id=f"fig-{k}", config=CFG, auto_play=False) for k, v in FIGS.items()}
sections = "".join(f'<section class="panel" id="s-{k}"><h2>{TITLES[k]}</h2><p class="how">{HOW[k]}</p>{divs[k]}<p class="cap"><b>What it shows.</b> {CAP[k]}</p></section>\n' for k in FIGS)
cards = "".join(f'<div class="card"><div class="big">{a}</div><div class="lab">{b}</div><div class="small">{c}</div></div>' for a, b, c in CARDS)
options = '<option value="">None (all groups equal)</option>' + "".join(f'<option value="{g}"{" selected" if g == HL_GROUP else ""}>{g}</option>' for g in show)
excl_txt = "; ".join(f"{g} ({why})" for g, why in excluded.items())

PAGE = """<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Indian airlines after COVID-19: who got their passengers back?</title>
<style>
:root{--hl:#d95f02;--dark:#404040;--mid:#8c8c8c;--bg:#f4f4f2}
*{box-sizing:border-box}body{margin:0;font-family:system-ui,-apple-system,"Segoe UI",Roboto,Arial,sans-serif;background:var(--bg);color:#222;line-height:1.5}
.wrap{max-width:1120px;margin:0 auto;padding:0 18px}
header{background:#fff;border-bottom:1px solid #ddd;padding:26px 0 10px}
h1{font-size:clamp(1.45rem,3.2vw,2.1rem);margin:0 0 6px;line-height:1.2}.sub{color:#555;margin:0 0 4px}
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:14px;margin:20px 0 10px}
.card{background:#fff;border:1px solid #e0e0e0;border-radius:10px;padding:14px 16px}.card .big{font-size:1.85rem;font-weight:700;color:var(--hl);line-height:1.15}
.card .lab{font-weight:600}.card .small{color:#666;font-size:.88rem}
#picker{position:sticky;top:0;z-index:30;background:#fff;border-bottom:1px solid #ccc;padding:10px 0;box-shadow:0 2px 6px rgba(0,0,0,.06)}
#picker .row{display:flex;flex-wrap:wrap;gap:10px 18px;align-items:center}#picker select{font-size:1rem;padding:6px 8px;border:1px solid #999;border-radius:6px}
#picker .sentence{font-weight:600;flex:1 1 360px}#picker .mini{color:#444;font-size:.9rem}#picker .mini b{color:var(--hl)}
section.panel{background:#fff;border:1px solid #e0e0e0;border-radius:10px;padding:16px 20px 18px;margin:20px 0}
h2{margin:0 0 2px;font-size:1.3rem}.how{color:#555;font-size:.93rem;margin:.1rem 0 .6rem}
.cap{border-left:4px solid var(--hl);background:#fafafa;padding:.45rem .8rem;margin:.8rem 0 0}
.box{background:#fff;border:1px solid #e0e0e0;border-radius:10px;padding:14px 20px;margin:20px 0;font-size:.95rem}.box h3{margin:.2rem 0 .4rem}
footer{color:#555;font-size:.88rem;padding:10px 0 40px}a{color:#8a3f00}
</style>
<script>%%PLOTLYJS%%</script></head><body>
<header><div class="wrap"><h1>Which Indian airlines got their passengers back after COVID-19?</h1>
<p class="sub">Scheduled domestic flights, monthly data from the Directorate General of Civil Aviation (DGCA), January 2019 to %%LASTMONTH%%.</p>
<div class="cards">%%CARDS%%</div></div></header>
<div id="picker"><div class="wrap"><div class="row"><label for="pick"><b>Pick an airline group:</b></label><select id="pick">%%OPTIONS%%</select>
<span class="sentence" id="pk-sentence"></span></div>
<div class="row mini" id="pk-mini"></div></div></div>
<main class="wrap">%%SECTIONS%%
<div class="box"><h3>How we counted</h3><ul>
<li><b>Back to 2019</b> means the average of three months reached the average of the same three months of 2019, so seasons do not distort the comparison. The current position uses the latest three months.</li>
<li><b>Airline groups.</b> Air India, Air India Express, Vistara, AirAsia India and AIX Connect are combined as Air India Group for all years, and JetLite is counted with Jet Airways, so mergers do not look like collapses.</li>
<li><b>Stopped airlines</b> count as zero passengers after their last month with traffic.</li>
<li><b>Fair comparison only.</b> A group needs passengers in all 12 months of 2019 and at least 0.5% of that year's passengers. Not compared: %%EXCLUDED%%.</li>
<li><b>Data ends %%LASTMONTH%%.</b> The next month is left out because the airlines do not add up to DGCA's own total yet. DGCA labels its figures provisional.</li></ul></div>
</main>
<footer><div class="wrap"><b>Sources.</b> DGCA monthly traffic statistics (Excel files), <a href="https://www.dgca.gov.in/digigov-portal/?baseLocale=en_US?page=259/4184/servicename">dgca.gov.in</a>, collected %%COLLECTED%%.
Second copy for cross-checking: <a href="https://github.com/Vonter/india-aviation-traffic">Vonter/india-aviation-traffic</a> (commit %%COMMIT%%), the same DGCA files parsed separately. Charts: Plotly.js; data handling: pandas. Licences, terms of use and the AI tool used are listed in the project report.</div></footer>
<script>
const STATS = %%STATS%%; const HLC = "#d95f02", MIDC = "#8c8c8c";
const fmt = (v, s) => (v === null || v === undefined) ? "n/a" : v + s;
function highlight(name, tries) {
  tries = tries || 0; let pending = false;
  ["fig-recovery", "fig-lf"].forEach(id => {
    const gd = document.getElementById(id);
    if (!gd || !gd.data) { pending = true; return; }
    const idx = [], col = [], wid = [], opa = [];
    gd.data.forEach((t, i) => { if (t.meta && t.meta.kind === "line") { const on = t.meta.group === name;
      idx.push(i); col.push(on ? HLC : MIDC); wid.push(on ? 4 : (name ? 1.6 : 2.2)); opa.push(on || !name ? 1 : 0.55); } });
    if (idx.length) Plotly.restyle(gd, {"line.color": col, "line.width": wid, "opacity": opa, "textfont.color": col}, idx);
  });
  if (pending && tries < 20) setTimeout(() => highlight(name, tries + 1), 300);
}
function pick(name) {
  const s = STATS[name], sent = document.getElementById("pk-sentence"), mini = document.getElementById("pk-mini");
  if (!s) { sent.textContent = "Showing every group in the same grey."; mini.innerHTML = ""; highlight(""); return; }
  sent.textContent = name + ": " + s.sentence;
  mini.innerHTML = "<span>Passengers a month now: <b>" + s.now_m.toFixed(2) + " M</b>" + (s.avg19_m !== null ? " (2019: " + s.avg19_m.toFixed(2) + " M)" : "") + "</span>" +
    "<span>Share of all passengers: <b>" + s.share_now + " in 100</b> (2019: " + s.share19 + ")</span>" +
    "<span>Seats filled: <b>" + fmt(s.lf_now, " in 100") + "</b> (2019: " + fmt(s.lf19, "") + ")</span>";
  highlight(name);
}
document.getElementById("pick").addEventListener("change", e => pick(e.target.value));
window.addEventListener("load", () => pick(document.getElementById("pick").value));
</script></body></html>"""
safe = lambda s: s.replace("</", "<\\/")
html = (PAGE.replace("%%PLOTLYJS%%", get_plotlyjs()).replace("%%CARDS%%", cards).replace("%%OPTIONS%%", options).replace("%%SECTIONS%%", sections)
            .replace("%%EXCLUDED%%", excl_txt).replace("%%LASTMONTH%%", f"{LAST:%B %Y}").replace("%%COLLECTED%%", meta["collection_date"])
            .replace("%%COMMIT%%", meta["github_commit"][:7]).replace("%%STATS%%", safe(json.dumps(stats))))
OUT_FILE = OUT / f"dgca_dashboard_{TODAY}.html"
OUT_FILE.write_text(html, encoding="utf-8")
print(f"saved {OUT_FILE.name}: {OUT_FILE.stat().st_size / 1e6:.1f} MB (works offline: Plotly is embedded)")
for k in CAP: print(f"\n[{k}] {CAP[k]}")

saved dgca_dashboard_2026-11-03.html: 5.5 MB (works offline: Plotly is embedded)

[market] In 2019 about 12.0 million people a month took a scheduled domestic flight. In Apr 2020 that fell to zero, and the market first matched its 2019 level again in Feb 2023. Over the last 12 months it averaged 13.9 million a month, 116% of 2019.

[recovery] IndiGo now carries 8.3 million passengers a month against 5.7 million in 2019 (146%), and Air India Group 3.1 million against 2.7 million (111%). SpiceJet is at 11% of 2019 and Go First stopped flying after May 2023.

[race] In August 2026, 2 of the 6 groups that flew in the same month of 2019 are above their black tick (IndiGo, Air India Group). Go First's bar vanishes after May 2023. Akasa Air, which began in August 2022, has no 2019 tick, and Star Air's tick is its launch year, so neither is counted.

[share] IndiGo carried 47 of every 100 passengers in 2019 and 66 of every 100 in the latest three months. Air India Group moved from 23 to 25, Sp

## 5. Open it

In [6]:
import webbrowser
webbrowser.open(OUT_FILE.resolve().as_uri())
print(OUT_FILE)

/Users/pranauvmac/Programming/Acads_Python/DA2402_DCV/DA2402-Project/outputs/dashboard/dgca_dashboard_2026-11-03.html
